# CS 229 - Homework 1: Alien CalcGPT

Submit **PDF** of completed notebook on Canvas.

**Maximum points**: 15

<div style="margin-bottom: 15px; padding: 15px; color: #31708f; background-color: #d9edf7; border: 1px solid #bce8f1; border-radius: 5px;">

<b><font size=+2>Enter your information below:</font></b><br/><br/>

  <b>(full) Name</b>: Vedant Deepak Borkute
  <br/>
  <b>Student ID Number</b>:  862552981
  <br/><br/>

<b>By submitting this notebook, I assert that the work below is my own work, Except where explicitly cited, none of the portions of this notebook are duplicated from anyone else's work.</b>
</div>

# Overview

In this assignment you will compare different methods for adapting a language model to understand math problems written in an **alien symbolic language**.

The alien language transforms standard arithmetic into unfamiliar syntax:

| Alien | Meaning | Example |
|-------|---------|--------|
| `flarn(a, b)` | a + b | `flarn(3, 5)` = 8 |
| `trok(a, b)` | a * b | `trok(4, 3)` = 12 |
| `snib(a, b)` | a - b | `snib(7, 2)` = 5 |
| `glorp(x)` | return x | `glorp(flarn(3, 5))` = 8 |

You will implement **five methods** and compare their effectiveness:
1. Base Prompting — minimal instruction
2. System Prompting — define the alien operators
3. Chain-of-Thought — step-by-step reasoning
4. In-Context Learning — provide solved examples
5. LoRA Fine-Tuning — adapt the model weights

> You are not being evaluated on finding the best prompt, but rather on correct implementation.

Complete all sections marked `TODO` and answer the analysis questions in Part 6.

## Setup
Run these cells to load the model and data. **Do not modify.**

In [69]:
# Install dependencies (run this on Colab; skip if already installed locally)
# !pip install -q transformers torch peft accelerate tqdm

import torch
import re
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [70]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME}")

Using device: cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct


## Load Data

Download `hw1_data.pt` from Canvas and place it in the same directory as this notebook.

The dataset contains arithmetic problems in alien notation, split into train (300) and test (150) sets at three difficulty levels:
- **Level 1**: single operation, e.g. `glorp(flarn(3, 5))`
- **Level 2**: one nested operation, e.g. `glorp(trok(4, flarn(2, 3)))`
- **Level 3**: two nested operations, e.g. `glorp(flarn(trok(2, 3), snib(10, 4)))`

In [71]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
data = torch.load('drive/MyDrive/HW 1 Alien CalcGPT.pt', weights_only=False)

In [73]:
# data = torch.load('hw1_data.pt', weights_only=False)
train_problems  = data['train_problems']
train_answers   = data['train_answers']
train_levels    = data['train_levels']
train_standard  = data['train_standard']   # standard math notation, e.g. "3 + 5"
test_problems   = data['test_problems']
test_answers    = data['test_answers']
test_levels     = data['test_levels']
test_standard   = data['test_standard']    # standard math notation, e.g. "3 + 5"
operators       = data['operators']

print(f"Train: {len(train_problems)} problems")
print(f"Test:  {len(test_problems)} problems")
print(f"\nAlien operators:")
for name, desc in operators.items():
    print(f"  {name}(...) = {desc}")
print(f"\nExample problems (one per difficulty level):")
for i in [0, 100, 200]:
    print(f"  Level {train_levels[i].item()}: {train_problems[i]}")
    print(f"  {'':>10}Standard math: {train_standard[i]} = {train_answers[i].item()}")

Train: 300 problems
Test:  150 problems

Alien operators:
  flarn(...) = addition (+)
  trok(...) = multiplication (*)
  snib(...) = subtraction (-)
  glorp(...) = return result (identity wrapper)

Example problems (one per difficulty level):
  Level 1: glorp(snib(8, 5))
            Standard math: 8 - 5 = 3
  Level 2: glorp(flarn(flarn(17, 10), 7))
            Standard math: (17 + 10) + 7 = 34
  Level 3: glorp(trok(trok(3, 4), flarn(1, 4)))
            Standard math: (3 * 4) * (1 + 4) = 60


## Helper Functions (Provided)

These functions handle model generation, answer extraction, and evaluation. **Do not modify** — your code will use them.

- `generate_response(messages, model, tokenizer)` — sends chat messages to the model and returns the text response
- `extract_answer(response)` — extracts the integer after `Final Answer:`
- `run_method(name, prompt_fn, problems, model, tokenizer)` — runs your prompt function on all test problems
- `print_results(...)` — prints accuracy overall and by difficulty level

In [74]:
def generate_response(messages, model, tokenizer, max_new_tokens=64):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def extract_answer(response):
    """Extract the integer after 'Final Answer:' in a response string."""
    match = re.search(r"Final Answer:\s*(-?\d+)", response)
    return int(match.group(1)) if match else None


def run_method(name, prompt_fn, problems, model, tokenizer, max_new_tokens=64):
    """Run a prompting method on all problems; return (predictions, raw_outputs)."""
    predictions, raw_outputs = [], []
    for problem in tqdm(problems, desc=name):
        response = generate_response(prompt_fn(problem), model, tokenizer, max_new_tokens)
        predictions.append(extract_answer(response))
        raw_outputs.append(response)
    return predictions, raw_outputs


def print_results(name, predictions, answers, levels):
    """Print accuracy overall and by difficulty level. Returns overall accuracy."""
    correct = sum(1 for p, a in zip(predictions, answers) if p == a.item())
    total = len(answers)
    acc = correct / total
    print(f"\n{'=' * 55}")
    print(f"  {name}")
    print(f"  Overall: {correct}/{total} ({100*acc:.1f}%)")
    for level in [1, 2, 3]:
        idxs = [i for i in range(total) if levels[i].item() == level]
        lc = sum(1 for i in idxs if predictions[i] == answers[i].item())
        print(f"  Level {level}: {lc}/{len(idxs)} ({100*lc/len(idxs):.1f}%)")
    n_fail = sum(1 for p in predictions if p is None)
    if n_fail:
        print(f"  Format failures: {n_fail}")
    print(f"{'=' * 55}")
    return acc


# Dict to collect results from all methods for comparison plots
all_results = {}

In [75]:
# Quick demo: test the generation helper on a plain English math problem
demo = generate_response(
    [{"role": "user", "content": "What is 3 + 5? Respond with: Final Answer: <number>"}],
    model, tokenizer
)
print(f"Demo response: {demo}")
print(f"Extracted: {extract_answer(demo)}")

Demo response: Final Answer: 8
Extracted: 8


## Extra Credit

Extra credit is submitted separately on Canvas using the **HW 1 (EC)** assignment. You must submit a PDF (made with LaTeX) outlining the problem, your approach, results (including tables or figures), and a discussion of results. Also submit your code (.py or .ipynb). Extra credit is graded on quality, generally 1–5 points, though could be higher for novel research directions. If you have a direction, feel free to pitch it!

### EC1: Robustness & Variance
Run the same prompts multiple times with `do_sample=True` and `temperature > 0`. Measure the variance in answers and consistency across the different methods. How stable are the results? Which methods are most/least reliable?

For a deeper study, analyze using the methods in the ["hot mess" paper](https://arxiv.org/abs/2601.23045).

### EC2: Confidence Calibration
Examine the confidence calibration of all methods. The goal is to get `p(final answer | prompt, output reasoning)` for each prediction. You can compute this by running a forward pass with `model()` on the generated output and extracting the logits for the answer tokens. Use `torch.softmax` to convert logits to probabilities.

If all probabilities are low, try normalizing over a restricted answer set: compute `p(final answer = i)` for `i = 0, 1, ..., 500` and renormalize. This is a common technique for structured outputs.

Use 10 confidence bins to build a reliability diagram and compute the Expected Calibration Error (ECE) as described in [Guo et al., 2017](https://arxiv.org/abs/1706.04599).

### EC3: Adversarial Alien Obfuscation
Compare different ways of obfuscating the math. For instance, if we use something semantic like "smash" instead of "glorp" to represent addition, is it easier or harder? Can you design strings of characters that reliably prevent learning at all?

See [AutoPrompt (Shin et al., EMNLP 2020)](https://aclanthology.org/2020.emnlp-main.346.pdf) for methods to automatically search for such prompts.

### EC4: Discrete Diffusion
Repeat the homework using [LLaDA](https://arxiv.org/abs/2502.09992), a discrete diffusion language model, and report any differences in performance. LLaDA allows sampling using fewer steps — does this dramatically affect any of the strategies (maybe all of them)?

### EC5: Reinforcement Learning with Verifiable Rewards
Instead of supervised fine-tuning (LoRA), try using reinforcement learning to train the model. Since our alien math problems have verifiable correct answers, we can use the answer as a reward signal. Implement Group Relative Policy Optimization (GRPO) as described in the [DeepSeekMath paper (Shao et al., 2024)](https://arxiv.org/abs/2402.03300). Compare the RL-trained model's accuracy against the LoRA supervised approach.

### EC5: Reinforcement Learning with Verifiable Rewards (GRPO)

In this section, we will use Group Relative Policy Optimization (GRPO) to train the model using the correct answer as a verifiable reward signal.

In [76]:
# Install TRL, datasets, and upgrade pyarrow to fix binary incompatibility
# ! pip install -q -U trl datasets pyarrow

In [77]:
# ! pip install git+https://github.com/huggingface/trl.git

In [78]:
# ! pip install -U wandb
import os
os.environ["WANDB_DISABLED"] = "true"

In [79]:

# ! pip show wandb tsc

In [ ]:
import re
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

def prepare_rl_data(problems, answers):
    data = {
        "prompt": [
            [{"role": "user", "content": f"Solve: {p}\nRespond with: Final Answer: <number>"}] 
            for p in problems
        ],
        "answer": [str(a.item()) for a in answers]
    }
    return Dataset.from_dict(data)

train_rl_dataset = prepare_rl_data(train_problems, train_answers)

def exact_match_reward(completions, answer, **kwargs):
    rewards = []
    for completion, target in zip(completions, answer):
        if isinstance(completion, list):
            text = completion[0]["content"] if completion else ""
        else:
            text = completion
        match = re.search(r"Final Answer:\s*(-?\d+)", text)
        if match and match.group(1) == target:
            rewards.append(1.0) 
        else:
            rewards.append(0.0) 
    return rewards

print(f"Prepared RL Dataset with {len(train_rl_dataset)} examples.")

Prepared RL Dataset with 300 examples.


In [81]:
train_rl_dataset

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 300
})

In [82]:
# ! pip install --upgrade torchao

In [83]:
# view an example of the prompts and answers in the RL dataset
train_rl_dataset.features['prompt'], train_rl_dataset.features['answer']

(List({'role': Value('string'), 'content': Value('string')}), Value('string'))

In [84]:
from peft import LoraConfig, TaskType


In [85]:
rl_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type=TaskType.CAUSAL_LM)

training_args = GRPOConfig(
    output_dir="./rlvr_results",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=10,
    temperature=0.7,
    num_generations=4,
)

trainer = GRPOTrainer(
    model=rl_model,
    args=training_args,
    train_dataset=train_rl_dataset,
    reward_funcs=[exact_match_reward],
    peft_config=lora_config,
)

print("Starting RLVR (GRPO) training...")
trainer.train()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting RLVR (GRPO) training...


Step,Training Loss
10,0.048184
20,0.061625
30,0.000000
40,0.000000
50,-0.077392
60,0.040298
70,0.000000
80,0.000000
90,0.000000
100,0.015954


TrainOutput(global_step=900, training_loss=0.001711150506245237, metrics={'train_runtime': 511.2198, 'train_samples_per_second': 1.76, 'train_steps_per_second': 1.76, 'total_flos': 0.0, 'train_loss': 0.001711150506245237})

In [86]:
def build_rl_prompt(problem):
    """Prompt format used during RL training."""
    return [{"role": "user", "content": f"Solve: {problem}\nRespond with: Final Answer: <number>"}]

trainer.model.eval()
preds_rl, outputs_rl = run_method(
    "EC5. RLVR (GRPO)", build_rl_prompt, test_problems, trainer.model, tokenizer
)
acc_rl = print_results("EC5. RLVR (GRPO)", preds_rl, test_answers, test_levels)
all_results["EC5. RLVR"] = {
    "predictions": preds_rl, "raw_outputs": outputs_rl, "accuracy": acc_rl
}
for i in range(3):
    status = "CORRECT" if preds_rl[i] == test_answers[i].item() else "WRONG"
    print(f"Problem:  {test_problems[i]}")
    print(f"Expected: {test_answers[i].item()}  |  Got: {preds_rl[i]}  [{status}]")
    print(f"Output:   {outputs_rl[i][:200]}")
    print()


EC5. RLVR (GRPO): 100%|██████████| 150/150 [00:11<00:00, 12.70it/s]


  EC5. RLVR (GRPO)
  Overall: 12/150 (8.0%)
  Level 1: 10/50 (20.0%)
  Level 2: 1/50 (2.0%)
  Level 3: 1/50 (2.0%)
Problem:  glorp(flarn(9, 7))
Expected: 16  |  Got: 63  [WRONG]
Output:   Final Answer: 63

Problem:  glorp(snib(13, 10))
Expected: 3  |  Got: 130  [WRONG]
Output:   Final Answer: 130

Problem:  glorp(flarn(11, 16))
Expected: 27  |  Got: 20  [WRONG]
Output:   Final Answer: 20

Explanation:
The expression `glorp(flarn(11, 16))` involves the function `flarn`, which is not defined in the standard mathematical functions available to me. However, if we assume t



In [87]:
def show_failures(method_name, n=3):
    preds = all_results[method_name]["predictions"]
    outputs = all_results[method_name]["raw_outputs"]
    print(f"\n--- Sample failures: {method_name} ---")
    count = 0
    for i in range(len(preds)):
        if preds[i] != test_answers[i].item():
            print(f"  Problem:  {test_problems[i]}")
            print(f"  Standard: {test_standard[i]} = {test_answers[i].item()}")
            print(f"  Got:      {preds[i]}")
            print(f"  Output:   {outputs[i][:200]}")
            print()
            count += 1
            if count >= n:
                break
    if count == 0:
        print("  No failures!")

for m in all_results:
    show_failures(m)



--- Sample failures: EC5. RLVR ---
  Problem:  glorp(flarn(9, 7))
  Standard: 9 + 7 = 16
  Got:      63
  Output:   Final Answer: 63

  Problem:  glorp(snib(13, 10))
  Standard: 13 - 10 = 3
  Got:      130
  Output:   Final Answer: 130

  Problem:  glorp(flarn(11, 16))
  Standard: 11 + 16 = 27
  Got:      20
  Output:   Final Answer: 20

Explanation:
The expression `glorp(flarn(11, 16))` involves the function `flarn`, which is not defined in the standard mathematical functions available to me. However, if we assume t



## Submission

Export this notebook to PDF and submit on Canvas.
```python
!jupyter nbconvert --to pdf --output=yourname_hw1.pdf hw1.ipynb
```